In [25]:
from typing import Optional, Annotated, List
from pydantic import BaseModel, Field, EmailStr, HttpUrl, SecretStr, Field, ConfigDict, ValidationError
from pydantic import field_validator, model_validator, validate_call, computed_field
from pydantic_settings import BaseSettings
from uuid import uuid4, UUID
import re
from decimal import Decimal, ROUND_HALF_EVEN
from datetime import datetime
from enum import Enum
from sqlalchemy import Column, String, Boolean
from sqlalchemy.orm import declarative_base

In [ ]:
class UserProfile(BaseModel):
    id: UUID
    email: EmailStr
    name: str
    password: SecretStr
    website: Optional[HttpUrl] = None
    bio: Optional[str] = None

    @field_validator("name")
    @classmethod
    def normalize_name(cls, v: str) -> str:
        v = v.strip()
        v = re.sub(r'\s+', ' ', v)
        v = v.title()
        return v

    @field_validator("password")
    @classmethod
    def password_strength(cls, v: SecretStr) -> SecretStr:
        password_value = v.get_secret_value()
        if len(password_value) < 8:
            raise ValueError("Пароль должен быть не короче 8 символов")
        return v

    @model_validator(mode="after")
    def check_domains(self):
        if self.website is None:
            return self

        email_domain = self.email.split('@')[1].lower()
        website_host = self.website.host.lower()

        if website_host.startswith('www.'):
            website_host = website_host[4:]

        if email_domain == website_host:
            raise ValueError(
                f"Домен сайта ({website_host}) не должен совпадать "
                f"с доменом email ({email_domain})"
            )

        return self

In [ ]:
@validate_call
def place_order(
    user_id: UUID,
    sku: Annotated[str, Field(min_length=3, max_length=12)],
    quantity: Annotated[int, Field(gt=0)],
    price: Annotated[Decimal, Field(ge=0)]
) -> dict:
    if not re.match(r'^[A-Z0-9]+$', sku):
        raise ValueError(
            f"SKU должен содержать только заглавные буквы и цифры, получено: {sku}"
        )

    price = price.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN)
    amount = (price * quantity).quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN)

    return {
        "user_id": user_id,
        "sku": sku,
        "quantity": quantity,
        "price": price,
        "amount": amount
    }


In [13]:
SKU_RE = re.compile(r"^[A-Z0-9]{3,12}$")


class OrderStatus(str, Enum):
    new = "new"
    paid = "paid"
    delivered = "delivered"
    canceled = "canceled"


class OrderItem(BaseModel):
    sku: str
    qty: int
    unit_price: Decimal

    @field_validator("sku")
    @classmethod
    def sku_format(cls, v: str) -> str:
        if not SKU_RE.match(v):
            raise ValueError(
                f"SKU должен содержать только заглавные буквы и цифры "
                f"и иметь длину 3-12 символов, получено: {v}"
            )
        return v

    @field_validator("qty")
    @classmethod
    def qty_positive(cls, v: int) -> int:
        if v <= 0:
            raise ValueError(f"Количество должно быть больше 0, получено: {v}")
        return v

    @field_validator("unit_price")
    @classmethod
    def price_non_negative(cls, v: Decimal) -> Decimal:
        if v < 0:
            raise ValueError(f"Цена не может быть отрицательной, получено: {v}")
        return v.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN)

    @computed_field
    @property
    def subtotal(self) -> Decimal:
        return (self.unit_price * self.qty).quantize(
            Decimal('0.01'),
            rounding=ROUND_HALF_EVEN
        )


class Order(BaseModel):
    id: UUID
    user_email: EmailStr
    items: List[OrderItem]
    status: OrderStatus = OrderStatus.new
    created_at: datetime = Field(default_factory=datetime.utcnow)

    @computed_field
    @property
    def total(self) -> Decimal:
        if not self.items:
            return Decimal('0.00')

        total_sum = sum(item.subtotal for item in self.items)
        return total_sum.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN)

    @model_validator(mode="after")
    def check_business_rules(self):
        if not self.items:
            raise ValueError("Заказ не может быть создан с пустой корзиной")

        if self.status in (OrderStatus.paid, OrderStatus.delivered):
            if self.total == 0:
                raise ValueError(
                    f"Заказ не может быть переведен в статус '{self.status.value}' "
                    f"при нулевой сумме"
                )

        return self

In [19]:
class APISettings(BaseSettings):
    base_url: HttpUrl
    token: SecretStr
    timeout_sec: int = 5
    retries: int = 2

    @field_validator("timeout_sec", "retries")
    @classmethod
    def check_ranges(cls, v: int, info):

        field_name = info.field_name

        if field_name == "timeout_sec":
            if not (1 <= v <= 60):
                raise ValueError(
                    f"timeout_sec должен быть в диапазоне 1-60, получено: {v}"
                )
        elif field_name == "retries":
            if not (0 <= v <= 10):
                raise ValueError(
                    f"retries должен быть в диапазоне 0-10, получено: {v}"
                )

        return v

    model_config = ConfigDict(
        env_prefix="API_",
        env_file=".env",
        extra="ignore"
    )

In [21]:
Base = declarative_base()

class SAUser(Base):
    __tablename__ = "users"
    id = Column(String, primary_key=True, default=lambda: str(uuid4()))
    email = Column(String, nullable=False)
    is_active = Column(Boolean, default=True)

    def __init__(self, email: str, is_active: bool = True):
        self.id = str(uuid4())
        self.email = email
        self.is_active = is_active


class UserOut(BaseModel):
    id: UUID
    email: EmailStr
    is_active: bool

    @field_validator("id", mode="before")
    @classmethod
    def convert_id_to_uuid(cls, v):
        if isinstance(v, str):
            return UUID(v)
        return v

    model_config = ConfigDict(
        from_attributes=True
    )

In [27]:
ORDER_SCHEMA = Order.model_json_schema()


def safe_create_order(data: dict) -> tuple[bool, str]:
    try:
        order = Order.model_validate(data)
        return (True, f"<total={order.total}>")

    except ValidationError as e:
        errors = e.errors()
        if errors:
            first_error = errors[0]
            field_path = ".".join(str(loc) for loc in first_error['loc'])
            error_msg = first_error['msg']
            return (False, f"[{field_path}] {error_msg}")
        return (False, "Ошибка валидации")

    except Exception as e:
        return (False, f"{type(e).__name__}: {str(e)[:100]}")